# AI Research Assistant
This notebook utilizes **LangChain**, **Groq**, and **FAISS** to perform RAG-based research on academic papers from arXiv.

### How to use:
1. Ensure your `GROQ_API_KEY` is set in the Colab secrets sidebar.
2. Run all cells in order to initialize the environment and agent.
3. Use the interactive input box at the bottom of the notebook to query the assistant.

In [1]:
!pip install -U langchain langchain-openai langchain-community langchain-core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 3.7 MB/s eta 0:00:00


In [2]:
!pip install datasets sentence-transformers faiss-cpu scikit-learn transformers huggingface-hub keybert langchain langchain-community langchain-core langchain-groq

In [3]:
from datasets import load_dataset
dataset = load_dataset("CShorten/ML-ArXiv-Papers", split="train")
print(dataset)

Dataset({
    features: ['Unnamed: 0.1', 'Unnamed: 0', 'title', 'abstract'],
    num_rows: 117592
})


In [4]:
import pandas as pd
df = pd.DataFrame(dataset)
df = df[['title', 'abstract']]
df = df.head(15000)
df["paper_text"] = df["title"] + " " + df["abstract"]
df["paper_text"] = df["paper_text"].str.replace("\n", " ", regex=False).str.strip()

In [5]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [6]:
import faiss
import numpy as np

# Convert to float32 for FAISS
embeddings = np.array(model.encode(df["paper_text"].tolist())).astype('float32')
index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)

In [7]:
def search_papers(query, k=5):
      query_embedding = np.array(model.encode([query])).astype('float32')
      faiss.normalize_L2(query_embedding)
      D, I = index.search(query_embedding, k)
      return D, I

In [8]:
from transformers import pipeline

# 'summarization' ki jagah 'text2text-generation' ka use karein
summarizer = pipeline("text-generation", model="facebook/bart-large-cnn")

def summarize_text(text):
    # Bart-large-cnn text2text-generation mein bhi sahi kaam karta hai
        summary = summarizer(text, max_length=120, min_length=40, do_sample=False)
        return summary[0]['generated_text']

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/316 [00:00<?, ?it/s]

[transformers] BartForCausalLM LOAD REPORT from: facebook/bart-large-cnn
Key                                                       | Status     |  | 
----------------------------------------------------------+------------+--+-
model.encoder.layers.{0...11}.self_attn_layer_norm.weight | UNEXPECTED |  | 
model.encoder.layers.{0...11}.self_attn_layer_norm.bias   | UNEXPECTED |  | 
model.encoder.layers.{0...11}.self_attn.k_proj.bias       | UNEXPECTED |  | 
model.encoder.layers.{0...11}.fc1.bias                    | UNEXPECTED |  | 
model.encoder.layers.{0...11}.fc2.weight                  | UNEXPECTED |  | 
model.encoder.layers.{0...11}.self_attn.v_proj.bias       | UNEXPECTED |  | 
model.encoder.layers.{0...11}.self_attn.out_proj.weight   | UNEXPECTED |  | 
model.encoder.layers.{0...11}.self_attn.k_proj.weight     | UNEXPECTED |  | 
model.encoder.layers.{0...11}.final_layer_norm.bias       | UNEXPECTED |  | 
model.encoder.layers.{0...11}.self_attn.out_proj.bias     | UNEXPECTED |  | 
mod

In [9]:
from langchain_core.tools import tool
from keybert import KeyBERT

kw_model = KeyBERT(model)

@tool
def extract_keywords(text: str, top_n: int = 5) -> str:
    """Extract the most important keywords from the given text using the KeyBERT model."""
    keywords = kw_model.extract_keywords(text, keyphrase_ngram_range=(1, 2), stop_words="english", top_n=top_n)
    result = "Top Keywords:\n\n"
    for rank, (keyword, score) in enumerate(keywords, start=1):
            result += f"{rank}. {keyword} (Relevance Score: {round(score, 4)})\n"
            return result

In [26]:
def search_and_summarize(query: str):
      try:
              # Assuming search_client is your arXiv search object
              search_results = search_client.query(query)
              papers = []

              for result in search_results:
                                                          paper_data = {
                                                                          "title": result.title,
                                                                         "summary": result.summary
                                                                       }
                                                                                                                  # THIS is where the indentation fix happens:
              papers.append(paper_data)

              return str(papers)
      except Exception as e:
              return f"An error occurred while searching: {str(e)}"

In [11]:
@tool
def compare_papers(paper1: str, paper2: str) -> str:
    """Compare two research papers based on their abstracts."""
        # Logic for finding and comparing
    embedding1 = model.encode([paper1])
    D1, I1 = index.search(np.array(embedding1).astype('float32'), 1)
    first_paper = df.iloc[I1[0][0]]

    embedding2 = model.encode([paper2])
    D2, I2 = index.search(np.array(embedding2).astype('float32'), 1)
    second_paper = df.iloc[I2[0][0]]

    comparison_prompt = f"Compare the following two research papers: {first_paper['title']} and {second_paper['title']}."
    return llm.invoke(comparison_prompt).content

In [12]:
from langchain_groq import ChatGroq
from google.colab import userdata
from langchain.agents import create_agent

# API Key setup: Google Colab ke 'Secrets' mein 'GROQ_API_KEY' set karein
api_key = userdata.get("GROQ_API_KEY")
llm = ChatGroq(model="llama-3.1-8b-instant", api_key=api_key)

tools = [search_and_summarize, extract_keywords, compare_papers]

In [13]:
!pip install --upgrade langchain langchain-core langchain-community

In [27]:
# Final Execution Cell
print("--- Research Assistant Ready ---")
print("Type 'exit' to stop.")

while True:
    user_query = input("\nWhat would you like to know? ")
    if user_query.lower() == 'exit':
                break

    try:
                                # This calls your agent executor
                response = agent_executor.invoke({"input": user_query})
                print("\nAssistant:", response['output'])
    except Exception as e:
                print(f"\nSomething went wrong: {e}")

--- Research Assistant Ready ---
Type 'exit' to stop.

What would you like to know? exit
